# ✅ Solutions — Chapter 5 — Methods and Methodologies — agentic lab

This is the **solution** notebook: the same lab with all 3 tasks worked. It runs clean end to end, which is what proves the reference implementations satisfy the marking scheme.

> Student version: [`04_agentic_lab.ipynb`](04_agentic_lab.ipynb)

# Chapter 5 — Methods and Methodologies
### Notebook 4 · Agentic lab — auditing, and planning under prerequisites

*Book reference: Extends §5.1–5.2*

An auditor agent that recommends a methodology and finds OntoClean violations, plus the course's first **planning MDP**: actions with prerequisites, rewarded by measured competency-question coverage.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch05_toolkit as ch5
from oe_course.sparql import SparqlStore
from oe_course.data import corpus
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [ ]:
import ch05_agentic as AG
from oe_course import evaluation as ev, llm, mdp, optimize as opt
import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

**By the end of this notebook you can:**

1. Build an agent for a task where **a reasoner is useless** — the errors are ontological, not logical.
2. Plan a project as an MDP with **precedence constraints**, where skipping a step makes later steps unavailable.
3. Diagnose a **second-order** optimisation failure: a rule that only becomes visible after another rule is learned.
4. Recognise a reward model that rewards the wrong plan — and fix it.

> **Prerequisite:** the Chapter 1 agentic lab.

## 1. Tools

Note what is *not* here: a reasoner. Every taxonomy in this lab is consistent, so a tableau would return "fine" on all of them. The tools an agent needs here are meta-property lookups and constraint checks.

In [ ]:
ctx = AG.Ch5Context()
tools = {t.name: t for t in AG.build_toolset(ctx)}
for name, t in tools.items():
    print(f'{name:30s} {list(t.args_schema.model_json_schema().get("properties", {}))}')
    print(f'{"":30s} {t.description.splitlines()[0]}')

In [ ]:
print(tools['ontoclean_tags'].invoke({'class_name': 'Student'}))
print(tools['ontoclean_tags'].invoke({'class_name': 'Person'}))
print()
print(tools['check_taxonomy'].invoke({'axioms': 'Person <= Student\nStudent <= Person'}))
print('\ntrajectory:', ctx.log.names())

### The tools also drive a project

`perform_step` refuses to run a step whose prerequisites are missing — the same precedence structure the MDP formalises below.

In [ ]:
ctx2 = AG.Ch5Context()
t2 = {t.name: t for t in AG.build_toolset(ctx2)}
print('try axioms first :', t2['perform_step'].invoke({'step': 'axioms'}))
for step in ['requirements', 'competency_questions', 'taxonomy', 'axioms']:
    print(f'{step:22s}', t2['perform_step'].invoke({'step': step}))

## 2. The dataset

Ten cases, each a brief plus a taxonomy. The two halves are independent so the metric can say *which* half an agent is failing. The split is stratified so both halves cover all five methodologies and both clean and violating taxonomies.

In [ ]:
all_cases = AG.build_dataset('all')
print(pd.DataFrame([{'id': e.id, 'methodology': e.gold_methodology,
                     'violations': len(e.gold_violations)} for e in all_cases]
                   ).to_string(index=False))
train, dev = AG.build_dataset('train'), AG.build_dataset('dev')
print('\ntrain methodologies:', sorted({e.gold_methodology for e in train}))
print('dev   methodologies:', sorted({e.gold_methodology for e in dev}))

In [ ]:
example = dev[0]
print('brief   :', example.brief)
print('taxonomy:')
print('  ' + example.taxonomy.replace('\n', '\n  '))
print('gold    :', example.gold_methodology, '|', example.gold_violations)

## 3. Baseline and GEPA

The un-instructed agent does what an inexperienced engineer does: names METHONTOLOGY because it is the one everyone has heard of, and reports no violations because the axioms all look fine — which, logically, they are.

In [ ]:
lm = llm.configure_dspy(AG.AUDIT_RULEBOOK, AG.audit_responder)
baseline = AG.AuditProgram()
pred = baseline(**example.inputs())
print('recommended:', pred.methodology, '| violations:', pred.violations)
report = AG.audit_scorer(example, pred)
print('score      :', report.score)
for n in report.notes:
    print('   ', n)

In [ ]:
before = ev.evaluate_dataset(baseline, dev, AG.audit_scorer)
print('BEFORE:', before['mean_score'])
print('violations:', before['violations'])

In [ ]:
gepa_metric = ev.make_gepa_metric(AG.audit_scorer, AG.AUDIT_RULEBOOK)
reflect = llm.reflection_lm(AG.AUDIT_RULEBOOK, AG.audit_responder)
tuned = opt.run_gepa(baseline, train, gepa_metric, valset=train,
                     max_metric_calls=90, reflection_lm=reflect)
result = opt.compare(AG.AuditProgram(), tuned, dev, AG.audit_scorer)
print(result.report())

## 4. A rule is learnable only if the data lets the agent break it

All six rules were discovered here. That is worth examining, because one of them very nearly could not have been.

`only-report-real-violations` punishes flagging a **sound** axiom as a violation. An agent can only commit that error if a sound axiom is present to be mis-flagged. Two of the training taxonomies deliberately mix a violating axiom with a sound one for exactly this reason — and if they did not, the rule would be unlearnable no matter how large the budget.

In [ ]:
found = AG.AUDIT_RULEBOOK.active_in(result.instruction_after)
print('rules discovered:', sorted(found))
print('rules missed    :', sorted(set(AG.AUDIT_RULEBOOK.ids) - found) or 'none')
print()
for e in train:
    axioms = e.taxonomy.splitlines()
    sound = [a for a in axioms if a.strip() not in e.gold_violations]
    print(f'  {e.id:22s} {len(axioms)} axioms, {len(sound)} sound '
          f'-> over-reporting {"possible" if sound and e.gold_violations else "impossible"}')

> **The general principle**, which is easy to state and easy to forget:

> *A failure mode your evaluation data makes impossible is a failure mode your agent will keep in production.*

Exercise 4.1 removes the sound axioms and shows the rule disappearing.

## 5. Planning as an MDP with prerequisites

Every earlier MDP made all actions available at all times. This one does not:

| | |
|---|---|
| **S** | which development steps are done, and whether we shipped |
| **A** | perform a step **whose prerequisites are complete**, or ship |
| **T** | deterministic |
| **R** | −effort per step; on ship, the **measured** CQ coverage |

Precedence is the structural claim every methodology in §5.1 makes. Here it is enforced by the action set, and the reward comes from the coverage table you built in Notebook 1 — not from a stipulated number.

In [ ]:
M = AG.MethodologyPlanMDP()
print(f'|S| = {len(M.states())}')
s0 = M.initial_state()
print('actions available at the start:', M.actions(s0))
print('\nNote what is NOT available: you cannot start with axioms or evaluation.')

In [ ]:
V, pi = mdp.value_iteration(M)
print(f'V*(s0) = {V[s0]:.3f}\n')
ep = mdp.run_episode(M, mdp.greedy_policy(pi))
for t in ep.transitions:
    print(f'  {str(t.state):10s} {t.action:22s} r={t.reward:+.2f}')
print(f'\noptimal plan: {" -> ".join(ep.actions)}')
print(f'return = {ep.discounted_return():.3f} '
      f'(coverage 1.0 minus {round(1.0 - ep.discounted_return(), 2)} of effort)')

> **Now look at what the optimal plan skips.** It never does `reuse_search`, and — more uncomfortably — it never does `evaluation`. Both cost effort and neither unlocks a competency question, so under *this* reward they are pure loss.

That is not a bug in the solver. It is a **bug in the reward model**, and it is the same bug that makes real teams skip evaluation under deadline pressure: the measured objective does not credit it. Exercise 4.2 asks you to fix the reward rather than the plan.

In [ ]:
skipped = [s for s in M.names if s not in ep.actions]
print('steps the optimal plan skips:', skipped)
for s in skipped:
    spec = ch5.DEVELOPMENT_STEPS[s]
    print(f'  {s:16s} cost={spec["cost"]:.2f} unlocks={spec["unlocks"] or "(nothing)"}')

### Task 4.1 — Make a rule unlearnable

Strip every **sound** axiom out of the training taxonomies, so over-reporting becomes impossible on that data. Re-run GEPA and show `only-report-real-violations` is no longer discovered — even though the dev set still punishes it.

> **Hint.** Keep only the axioms that appear in `gold_violations`.

In [ ]:
import dspy
ablated = []
for e in train:
    keep = [a for a in e.taxonomy.splitlines() if a.strip() in e.gold_violations]
    ablated.append(dspy.Example(
        brief=e.brief, taxonomy='\n'.join(keep) or e.taxonomy,
        gold_methodology=e.gold_methodology, gold_violations=e.gold_violations,
        id=e.id).with_inputs('brief', 'taxonomy'))

print('ablated training taxonomies (violations only):')
for e in ablated:
    print(f'  {e.id:22s} {e.taxonomy.splitlines()}')

tuned_ablated = opt.run_gepa(AG.AuditProgram(), ablated, gepa_metric, valset=ablated,
                             max_metric_calls=90, reflection_lm=reflect)
res_ablated = opt.compare(AG.AuditProgram(), tuned_ablated, dev, AG.audit_scorer)
found_ablated = AG.AUDIT_RULEBOOK.active_in(res_ablated.instruction_after)

print('\nfull train    -> dev', result.after['mean_score'],
      '| rules', len(AG.AUDIT_RULEBOOK.active_in(result.instruction_after)))
print('ablated train -> dev', res_ablated.after['mean_score'],
      '| rules', len(found_ablated))
print('over-reporting rule learned?', 'only-report-real-violations' in found_ablated)
print('\nThe rule vanished. Nothing about the agent, the metric or the budget\n'
      'changed -- only the data. Two axioms removed from a training set silently\n'
      'removed a capability, and the only symptom is a dev score 0.03 lower.\n'
      'In production the symptom would be an auditor that cries wolf.')

**Checks for Task 4.1.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert 'only-report-real-violations' not in found_ablated

### Task 4.2 — Fix the reward so evaluation is worth doing

The optimal plan skips `evaluation`. Change the reward so that skipping it is penalised — for example, because unevaluated ontologies ship with defects — and show the optimal plan changing.

> **Hint.** Pass a custom `coverage_fn` to `MethodologyPlanMDP`.

In [ ]:
def coverage_with_evaluation(done):
    """Shipping without evaluation loses a fifth of the delivered value.

    The justification is empirical, not moral: unevaluated ontologies ship with
    defects that cost more to fix later than the evaluation would have cost.
    """
    base = ch5.coverage_for_steps(done)
    return base if 'evaluation' in done else base * 0.8

M2 = AG.MethodologyPlanMDP(coverage_fn=coverage_with_evaluation)
V2, pi2 = mdp.value_iteration(M2)
ep2 = mdp.run_episode(M2, mdp.greedy_policy(pi2))
print('new optimal plan:', ' -> '.join(ep2.actions))
print(f"V* = {V2[M2.initial_state()]:.3f} (was {V[s0]:.3f})")
print('\nEvaluation now pays for itself: 20 per cent of 1.0 coverage is 0.20, and the\n'
      'step costs 0.10. Nothing about the planner changed -- only what we told\n'
      'it to value. If your agents keep skipping something you care about, the\n'
      'first place to look is the reward, not the policy.')

**Checks for Task 4.2.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert 'evaluation' in ep2.actions

### Task 4.3 — Make reuse worth searching for

`reuse_search` is also skipped. Model the NeOn claim — that reuse *reduces the cost* of building the taxonomy and axioms — and find the discount at which searching for reusable resources becomes optimal.

In [ ]:
import copy
rows = []
for discount in [0.0, 0.2, 0.4, 0.6]:
    steps = copy.deepcopy(ch5.DEVELOPMENT_STEPS)
    # Modelling choice: reuse cannot make a step free, only cheaper.
    steps['taxonomy'] = dict(steps['taxonomy'],
                             requires=('competency_questions',))
    class ReuseMDP(AG.MethodologyPlanMDP):
        def transition(self, state, action):
            if action == 'ship':
                return [(1.0, type(state)(state.done, True), self.coverage(state.done))]
            cost = self.steps[action]['cost']
            if 'reuse_search' in state.done and action in ('taxonomy', 'axioms'):
                cost *= (1 - discount)
            return [(1.0, type(state)(state.done | {action}, False), -cost)]
    Mr = ReuseMDP(steps)
    Vr, pir = mdp.value_iteration(Mr)
    epr = mdp.run_episode(Mr, mdp.greedy_policy(pir))
    rows.append({'discount': discount, 'V*': round(Vr[Mr.initial_state()], 3),
                 'searches for reuse': 'reuse_search' in epr.actions})
print(pd.DataFrame(rows).to_string(index=False))
print('\nreuse_search costs 0.15 and can save at most discount x 0.45 (the cost of\n'
      'taxonomy + axioms), so it pays from roughly a third onwards. That is NeOn\n'
      'stated as an inequality: reuse is worth the search only when the resources\n'
      'you find genuinely displace work you would otherwise do.')

**Checks for Task 4.3.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert [r['discount'] for r in rows] == [0.0, 0.2, 0.4, 0.6]
# With no discount the search cannot pay for itself; with a large one it must.
assert not rows[0]['searches for reuse']
assert rows[-1]['searches for reuse']
# A discount can only make the plan cheaper.
assert rows[-1]['V*'] >= rows[0]['V*'] - 1e-9

## Chapter 5 in the course arc

| | Ch. 1 | Ch. 2 | Ch. 3 | Ch. 4 | Ch. 5 |
|---|---|---|---|---|---|
| MDP | gather evidence | search a proof | budgeted, stochastic | construct | **plan under prerequisites** |
| grader | labels + judge | decision procedure | free oracle | labels + profiles | measured CQ coverage |
| what a reasoner buys you | nothing | — | everything | everything | **nothing** |

Chapter 5's contribution: the errors that matter most are often the ones your tooling cannot see. A reasoner amplifies whatever you assert — including your mistakes — so a methodology and a meta-property checker are not bureaucracy, they are the only defence against a whole class of bug.

Chapter 6 takes the repair that OntoClean could not express (`Statue <= Clay` is *constitution*, not subsumption) and gives you the vocabulary for it.